# ⚠️ Lakebase 101 Demo — Full Cleanup

This notebook tears down **all** demo resources in order:
1. Synced table API registrations (Postgres project bindings)
2. Synced table pipelines
3. Delta tables
4. Schema + Catalog

**Run this only when you want to completely remove the demo.**

In [0]:
"""Delete synced table registrations from the Lakebase Postgres API.
This is the step that removes the project binding — without it, redeploying
the bundle against a new project fails with 'Table already exists pointing
to a different project'."""
from databricks.sdk import WorkspaceClient

CATALOG = "lakebase_101_catalog"
SCHEMA = "lakebase_101_schema"
w = WorkspaceClient()

synced_table_ids = [
    f"{CATALOG}.{SCHEMA}.customer_360_synced",
    f"{CATALOG}.{SCHEMA}.customers_directory_synced",
    f"{CATALOG}.{SCHEMA}.sales_events_synced",
]

print("Deleting synced table API registrations...")
for st_id in synced_table_ids:
    try:
        w.postgres.delete_synced_table(name=st_id)
        print(f"  🗑️  {st_id}")
    except Exception as e:
        print(f"  ⚠️  {st_id}: {e}")

print("\n✅ Synced table API registrations cleaned up.")

In [0]:
%sql
DROP TABLE IF EXISTS lakebase_101_catalog.lakebase_101_schema.sales_events;
DROP TABLE IF EXISTS lakebase_101_catalog.lakebase_101_schema.customer_360_gold;
DROP TABLE IF EXISTS lakebase_101_catalog.lakebase_101_schema.customers_directory;
DROP TABLE IF EXISTS lakebase_101_catalog.lakebase_101_schema.products;

In [0]:
%sql
DROP SCHEMA IF EXISTS lakebase_101_catalog.lakebase_101_schema CASCADE;
DROP CATALOG IF EXISTS lakebase_101_catalog CASCADE;

In [0]:
"""Delete ALL synced-table pipelines for this demo.
These are auto-created by the synced tables service and not cleaned up by bundle destroy."""
from databricks.sdk import WorkspaceClient

CATALOG = "lakebase_101_catalog"
w = WorkspaceClient()

pipelines = list(w.pipelines.list_pipelines(filter=f"name LIKE '%{CATALOG}%'"))
print(f"Found {len(pipelines)} synced-table pipelines to delete...")

for p in pipelines:
    try:
        w.pipelines.delete(p.pipeline_id)
        print(f"  🗑️  {p.pipeline_id} | {p.name}")
    except Exception as e:
        print(f"  ⚠️  {p.pipeline_id}: {e}")

print(f"\n✅ Deleted {len(pipelines)} pipeline(s).")

Found 3 synced-table pipelines to delete...
  🗑️  56f4bd9b-b741-445d-9eb7-736189016f7e | Synced table: lakebase_101_catalog.lakebase_101_schema.sales_events_synced 3zkQP6
  🗑️  67238238-ee20-45cb-870f-929703803d25 | Synced table: lakebase_101_catalog.lakebase_101_schema.customers_directory_synced FMiSRN
  🗑️  b37a51cd-bffa-4b89-a50d-70b645b45c8d | Synced table: lakebase_101_catalog.lakebase_101_schema.customer_360_synced 3rzvJy

✅ Deleted 3 pipeline(s).


In [0]:
print("""
✅ Full cleanup complete!

Removed:
  - Synced table API registrations (project bindings)
  - Synced table pipelines
  - Delta tables (products, customers_directory, customer_360_gold, sales_events)
  - Schema: lakebase_101_catalog.lakebase_101_schema
  - Catalog: lakebase_101_catalog
""")


✅ Full cleanup complete!

Removed:
  - Delta tables (products, customers_directory, customer_360_gold, sales_events)
  - Schema: lakebase_101_catalog.lakebase_101_schema
  - Catalog: lakebase_101_catalog

